# Data preparation of hydro reservoir level filling and inflows derived from aggregation from ENTSO-E transparency

source: https://transparency.entsoe.eu/

Creates the following parsed datasets

- Hourly storage levels per country for selected year for setting storage start and end conditions (reservoir_level_'+year+'_hourly_entsoe_TP.csv) 
- Weekly storage levels per country for selected year for setting storage start and end conditions (reservoir_level_'+year+'_weekly_entsoe_TP.csv) 

Settings in next window

In [1]:
#download files again (yes/no)?
download = "no"

#set year for data creation
year = '2017'

In [2]:
import pysftp
import sys
import os
import pandas as pd
import numpy as np

In [3]:
#ENTSO-E sftp settings
host = "sftp-transparency.entsoe.eu"
password = "qhfWzbuxRkmmKb+"                
username = "jonas.savelsberg@unibas.ch"                
port = '22'

In [4]:
dir_out = "../parsed_data/"

In [5]:
fn_additional = "../additional_data.xlsx"
df_countries= pd.read_excel(fn_additional, sheet_name='Countries_EU', index_col="Country")
countries = list(df_countries.index)

In [6]:
#Connect to ENTSO-E Transparency FTP
#for this to work, pysftp 0.2.8 is needed!
path = '/TP_export/'
path_local = os.path.join(os.pardir,'source_data/') 

In [7]:
with pysftp.Connection(host=host, username=username, password=password) as sftp:
    print("Connection succesfully established.")

    # show list of files
    files = sftp.listdir('/TP_export/')   
    #print(files)

C:\Users\jsavelsberg\Anaconda3\lib\site-packages\pysftp\__init__.py:61: UserWarning: Failed to load HostKeys from C:\Users\jsavelsberg\.ssh\known_hosts.  You will need to explicitly load HostKeys (cnopts.hostkeys.load(filename)) or disableHostKey checking (cnopts.hostkeys = None).
  warnings.warn(wmsg, UserWarning)


SSHException: No hostkey for host sftp-transparency.entsoe.eu found.

## load data

In [8]:
#set paths and get file names
path_level = path+'AggregateFillingRateWaterReservoirs/'
path_level_local = path_local+'hydro_ENTSOE/reservoir_level_transparency/'
with pysftp.Connection(host=host, username=username, password=password) as sftp:
    print("Connection succesfully established.")
    # show list of files
    files = sftp.listdir(path_level)
    #download files
    if year != "":
        files = [i for i in files if year in i]

Connection succesfully established.


In [9]:
#download aggregated load data (ActualTotalLoad)
if download =="yes":
    with pysftp.Connection(host=host, username=username, password=password) as sftp:
        for file in files:
            sftp.get(path_level+file,path_level_local+file)
            print('Successfully downloaded file '+file)

In [10]:
#combine files to one data frame
df_level = pd.DataFrame()
for file in files:
    df_temp = pd.read_csv(path_level_local+file,
                          decimal=".",encoding="UTF-16LE",sep="\t",
                          parse_dates=True, index_col="DateTime").drop("areacode", axis=1)
    df_level = df_level.append(df_temp)
df_level = df_level[df_level.AreaTypeCode == "CTY"].drop(["AreaTypeCode",'Year','Month','Day','ResolutionCode','AreaName','UpdateTime','DeletedFlag'], axis=1).reset_index()
df_level = df_level.sort_values(by=['DateTime'])
df_level = df_level.rename(columns={"MapCode": "country", 'DateTime':'date','StoredEnergy':'MWh'})
#add timestamp for first hour for all countries for upsampling in next step
for country in countries:
    df_level = df_level.append(pd.DataFrame([[pd.Timestamp(str(np.int64(year)-1)+'-12-31'),country,float("NaN")]], columns=['date','country','MWh']), ignore_index=True)

df_level.head()

FileNotFoundError: [Errno 2] No such file or directory: '..\\source_data/hydro_ENTSOE/reservoir_level_transparency/2017_10_AggregateFillingRateWaterReservoirs.csv'

In [ ]:
#values are reported weekly so we have to resample to hourly values
#upsample with backwardsfill and select baseyear again
df_level_hourly = df_level.pivot(index='date',columns='country')[['MWh']].resample('H').fillna("bfill")
df_level_hourly = df_level_hourly[pd.DatetimeIndex(df_level_hourly.reset_index().date).year.astype(str) == year].stack()
df_level_hourly.tail()

In [ ]:
#we also export weekly values
df_level = df_level.set_index(['date','country'])

In [ ]:
df_level.to_csv(dir_out+'reservoir_level_'+year+'_weekly_entsoe_TP.csv', encoding="utf-8", index=True)
df_level_hourly.to_csv(dir_out+'reservoir_level_'+year+'_hourly_entsoe_TP.csv', encoding="utf-8", index=True)